# DSSAT Converter - Run Main Function

This notebook runs the standard DSSAT converter. It generates DSSAT input files and executes each simulation independently, without DSSAT sequence mode.

## 1. Locate the Repository and Import Modules

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
repo_root = next(
    (
        path
        for path in candidates
        if (path / "tests" / "data").exists() and (path / "src" / "modfilegen").exists()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(f"Could not locate the ModFileGen repository from cwd={cwd}.")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from modfilegen import GlobalVariables
from modfilegen.Converter.DssatConverter.dssatconverter import fetch_data_from_sqlite, main

print(f"Kernel cwd: {cwd}")
print(f"Repo root: {repo_root}")

/mnt/d/Mes Donnees/TCMP/github/ModFileGen/src/modfilegen/Converter/DssatConverter/dssatconverter.py:49: SyntaxWarning: invalid escape sequence '\d'
  res = re.findall("([-]?\d+[.]?\d+)[_]", d)


Kernel cwd: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/notebooks
Repo root: /mnt/d/Mes Donnees/TCMP/github/ModFileGen


## 2. Configure Paths

In [2]:
data_dir = repo_root / "tests" /  "data"
output_dir = repo_root / "tests" /  "output_dssat"
temp_dir = output_dir / "temp"

master_input_db = data_dir / "MasterInput.db"
models_dict_db = data_dir / "ModelsDictionaryArise.db"
cultivars_folder = data_dir / "cultivars" / "dssat"

output_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(parents=True, exist_ok=True)

n_threads = 6
n_parts = 1
# dt=0 keeps generated DSSAT folders for inspection. Set dt=1 to clean intermediate files after successful runs.
dt = 0
thirdyear = 0

print(f"Master Input DB exists: {master_input_db.exists()} -> {master_input_db}")
print(f"Models Dict DB exists: {models_dict_db.exists()} -> {models_dict_db}")
print(f"Cultivars folder exists: {cultivars_folder.exists()} -> {cultivars_folder}")
print(f"Output directory: {output_dir}")
print(f"Temp directory: {temp_dir}")

Master Input DB exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/MasterInput.db
Models Dict DB exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/ModelsDictionaryArise.db
Cultivars folder exists: True -> /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/cultivars/dssat
Output directory: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat
Temp directory: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat/temp


## 3. Preview Simulations

In [3]:
rows = fetch_data_from_sqlite(str(master_input_db))
print(f"Total simulations: {len(rows)}")

for index, row in enumerate(rows[:10], start=1):
    print(
        f"{index}. {row['idsim']} | "
        f"site={row.get('idPoint')} start={row.get('StartYear')}-{row.get('StartDay')} "
        f"end={row.get('EndYear')}-{row.get('EndDay')}"
    )

Total simulations: 9
1. 5.925_6.025_2000_Mgt1M0_135_2 | site=5.925_6.025 start=2000-105 end=2000-335
2. 5.925_6.025_2001_Mgt1M0_135_2 | site=5.925_6.025 start=2001-105 end=2001-335
3. 5.925_6.025_2002_Mgt1M0_135_2 | site=5.925_6.025 start=2002-105 end=2002-335
4. 5.925_6.025_2000_Mgt1M0_150_2 | site=5.925_6.025 start=2000-120 end=2000-350
5. 5.925_6.025_2001_Mgt1M0_150_2 | site=5.925_6.025 start=2001-120 end=2001-350
6. 5.925_6.025_2002_Mgt1M0_150_2 | site=5.925_6.025 start=2002-120 end=2002-350
7. 5.925_6.025_2000_Mgt1M0_155_2 | site=5.925_6.025 start=2000-125 end=2000-355
8. 5.925_6.025_2001_Mgt1M0_155_2 | site=5.925_6.025 start=2001-125 end=2001-355
9. 5.925_6.025_2002_Mgt1M0_155_2 | site=5.925_6.025 start=2002-125 end=2002-355


## 4. Set Global Variables

In [4]:
GlobalVariables["dbMasterInput"] = str(master_input_db)
GlobalVariables["dbModelsDictionary"] = str(models_dict_db)
GlobalVariables["directorypath"] = str(output_dir)
GlobalVariables["pltfolder"] = str(cultivars_folder)
GlobalVariables["nthreads"] = n_threads
GlobalVariables["dt"] = dt
GlobalVariables["parts"] = n_parts
GlobalVariables["tempDir"] = str(temp_dir)
GlobalVariables["thirdyear"] = thirdyear  
GlobalVariables["dailyoutput"] = 1  

print("GlobalVariables configured:")
for key, value in GlobalVariables.items():
    print(f"  {key}: {value}")

GlobalVariables configured:
  storeNumMinSimu: 0
  storeNumMaxSimu: 0
  storeKeyDataN: 0
  dbMasterInput: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/MasterInput.db
  dbModelsDictionary: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/ModelsDictionaryArise.db
  dbCelsius: 
  dt: 0
  ori_MI: 
  parts: 1
  tempDir: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat/temp
  package: 
  thirdyear: 0
  directorypath: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat
  pltfolder: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/data/cultivars/dssat
  nthreads: 6
  dailyoutput: 1


## 5. Run the DSSAT Converter

In [5]:
print("Starting DSSAT conversion...")
print("=" * 60)

try:
    main()
    print("\n" + "=" * 60)
    print("DSSAT conversion completed successfully.")
except Exception as error:
    print("\n" + "=" * 60)
    print(f"Error during DSSAT conversion: {error}")
    import traceback

    traceback.print_exc()
    raise

Starting DSSAT conversion...
Indexes created successfully!
📊 Total simulations to process: 9
Processing 6 chunks...
Using in-memory concatenation for 9 simulations


Iteration 0
Iteration 0
Iteration 0
Iteration 0
Iteration 0
Iteration 0
Iteration 1
Iteration 1
Iteration 1
 Written 9 rows to /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat/f788d439-e957-4ccc-be7a-2631a3a52b0e_dssat.csv
✅ Results saved to /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat/f788d439-e957-4ccc-be7a-2631a3a52b0e_dssat.csv
DSSAT total time: 2.46s
✅ 9 rows inserted into SummaryOutput.

DSSAT conversion completed successfully.


## 6. Inspect Generated Results

In [6]:
result_files = sorted(output_dir.glob("*_dssat.csv"), key=lambda path: path.stat().st_mtime)
print(f"DSSAT result files: {len(result_files)}")

if result_files:
    latest_result = result_files[-1]
    print(f"Latest result: {latest_result}")
    import pandas as pd

    display(pd.read_csv(latest_result).head())
else:
    print("No DSSAT result CSV found yet.")

DSSAT result files: 4
Latest result: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/output_dssat/7c4cde30-9ae6-4a4c-8730-ce45674e709b_dssat.csv


,Model,Idsim,Texte,index,Planting,Emergence,Ant,Mat,HDAT,DWAP,...,SRADA,DAYLA,CO2A,PRCP,ETCP,CumE,Transp,lon,lat,time
0,Dssat,5.925_6.025_2000_Mgt1M0_135_2,NaN,0,2000135.0,2000140.0,2000210.0,2000267.0,2000335.0,-99.0,...,14.8,12.1,369.6,1877.5,676.6,408.9,269.2,6.025,5.925,2000
1,Dssat,5.925_6.025_2001_Mgt1M0_135_2,NaN,0,2001135.0,2001140.0,2001208.0,2001265.0,2001335.0,-99.0,...,15.0,12.1,371.2,1476.0,663.6,382.1,282.9,6.025,5.925,2001
2,Dssat,5.925_6.025_2002_Mgt1M0_135_2,NaN,0,2002135.0,2002140.0,2002208.0,2002264.0,2002335.0,-99.0,...,14.6,12.1,373.3,1908.1,659.0,404.4,255.8,6.025,5.925,2002
3,Dssat,5.925_6.025_2000_Mgt1M0_150_2,NaN,0,2000150.0,2000155.0,2000226.0,2000283.0,2000350.0,-99.0,...,15.0,12.0,369.7,1758.8,664.4,377.9,288.3,6.025,5.925,2000
4,Dssat,5.925_6.025_2001_Mgt1M0_150_2,NaN,0,2001150.0,2001155.0,2001225.0,2001281.0,2001350.0,-99.0,...,15.1,12.0,371.3,1346.7,644.5,340.1,305.6,6.025,5.925,2001
